
# Lab 3 – Debugging a Sentiment Pipeline with DSPY

In this lab, you'll explore how modular pipelines make AI systems more **debuggable and reliable**.  
You’ll build and inspect a **three-step DSPY pipeline** that analyzes text about Tulane University and summarizes overall sentiment.  
You’ll also compare this pipeline to a **single-prompt model**, and analyze intermediate steps to see why modularity matters.



## 🎯 Learning Goals
By the end of this lab, you will:
- Understand what a pipeline is and why modular design helps debugging.
- Build a simple three-step DSPY sentiment pipeline:
  1. `extract_sentences_about_tulane`
  2. `annotate_sentiment`
  3. `summarize_sentiment`
- Use `dspy.inspect()` to see what prompt is sent to the model.
- Handle pipeline errors gracefully using `safe_run()`.
- **Log and interpret intermediate steps** to analyze how pipeline modularity improves transparency.


In [1]:
# @title 🔧 Lab 3 Setup (Run this first)
!git clone -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')
from course_utils import lab3_setup, build_sentiment_pipeline_demo, safe_run_demo, single_prompt_sentiment_summary

lab3_setup()
print("✅ Lab 3 setup complete.")

Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
✅ lab3_setup complete — scientific libraries ready, helper function loaded.
✅ Lab 3 setup complete.



## 💬 Pre-Lab Questions

1. Why is it useful to split a text analysis task into multiple smaller steps?  
2. What kinds of errors might occur in a multi-step system?  
3. Why might it be helpful to inspect what prompt is being sent to the model?



## 🧠 Scientific Question & Hypothesis

**Question:**  
If one pipeline step fails (for example, the sentiment annotator mislabels or the extractor misses a sentence), how does that affect the final sentiment summary?

**Hypothesis:**  
I expect that early-step errors (like missing a relevant sentence) will cause bigger downstream effects than later-step errors.


In [2]:
import dspy
import os
lm = dspy.LM("openai/gpt-4o-mini", api_key=os.environ["OPENAI_API_KEY"])
dspy.configure(lm=lm)

In [3]:
# @title Run the Sentiment Pipeline
from course_utils import build_sentiment_pipeline_demo

article = """
Tulane University announced a new scholarship program this week.
Some students praised the decision as forward-thinking.
Others criticized the administration for not increasing campus wages.
Overall, Tulane continues to inspire pride across New Orleans.
"""

pipeline = build_sentiment_pipeline_demo()
summary = pipeline(article)
print("🟢 Final Sentiment Summary:", summary)

🟢 Final Sentiment Summary: The sentiments reflect a balanced perspective, combining both neutrality and positivity. This suggests feelings of contentment and acceptance, indicating an overall optimistic outlook without strong emotional extremes.


In [5]:
# @title Inspect the Model Prompt
import dspy
annotator = dspy.Predict("sentence -> sentiment: str")
example_sentence = "Tulane's facilities are improving every year!"
_ = annotator(sentence=example_sentence)
dspy.inspect_history()





[2025-12-28T16:04:20.690711]

System message:

Your input fields are:
1. `sentence` (str):
Your output fields are:
1. `sentiment` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## sentence ## ]]
{sentence}

[[ ## sentiment ## ]]
{sentiment}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `sentence`, produce the fields `sentiment`.


User message:

[[ ## sentence ## ]]
Tulane's facilities are improving every year!

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
positive

[[ ## completed ## ]]







In [6]:
# @title Simulate Failures with safe_run()
from course_utils import safe_run_demo

# 1. Valid input (works fine)
result_ok = safe_run_demo("Tulane just launched a new AI program.")
# 2. Invalid input (missing text)
result_bad = safe_run_demo("")
# 3. Corrupted data (simulate crash)
result_fail = safe_run_demo("simulate_error")

print("✅ OK result:", result_ok)
print("⚠️ Failure result:", result_fail)

⚠️ Error in pipeline step: Empty input text
⚠️ Error in pipeline step: Simulated model crash
✅ OK result: {'status': 'ok', 'message': 'All pipeline steps succeeded'}
⚠️ Failure result: None


In [7]:
# @title Log and Compare Pipeline Results
interactions = []

texts = [
    "Tulane students are thrilled about the new dorms!",
    "Some faculty worry about workload increases.",
    "",
    "simulate_error"
]

for text in texts:
    result = safe_run_demo(text)
    interactions.append({
        "input": text[:40],
        "result": str(result),
        "status": "✅" if result else "⚠️"
    })

import pandas as pd
pd.DataFrame(interactions)

⚠️ Error in pipeline step: Empty input text
⚠️ Error in pipeline step: Simulated model crash


,input,result,status
0,Tulane students are thrilled about the n,"{'status': 'ok', 'message': 'All pipeline step...",✅
1,Some faculty worry about workload increa,"{'status': 'ok', 'message': 'All pipeline step...",✅
2,,None,⚠️
3,simulate_error,None,⚠️


In [8]:
# @title Compare Single Prompt vs Pipeline
from course_utils import build_sentiment_pipeline_demo, single_prompt_sentiment_summary

article = """
Tulane University announced a major renovation project.
Some alumni are excited, calling it a smart investment.
A few students are frustrated by the temporary noise and construction.
Overall, Tulane continues to grow in positive ways.
"""

pipeline = build_sentiment_pipeline_demo()
pipeline_summary = pipeline(article)

single_summary = single_prompt_sentiment_summary(article)

print("🧩 Pipeline Summary:", pipeline_summary)
print("🧱 Single-Prompt Summary:", single_summary)

🧩 Pipeline Summary: The sentiments expressed reflect a mix of emotions: there are instances of positivity, indicating moments of satisfaction or happiness, but they are balanced by a negative sentiment that suggests some challenges or discontent. Overall, the blend of neutral sentiment implies a complex but generally favorable outlook.
🧱 Single-Prompt Summary: The overall sentiment about Tulane University is positive. The announcement of a major renovation project has generated excitement among some alumni, who view it as a smart investment for the university's future. While there are some frustrations among students regarding the temporary noise and disruption caused by the construction, the general consensus is that Tulane is continuing to grow and improve.


In [9]:
# @title Add Intermediate Logging
from course_utils import build_sentiment_pipeline_demo

def logged_sentiment_pipeline(article, verbose=True):
    pipeline = build_sentiment_pipeline_demo(verbose=verbose)
    summary = pipeline(article)
    return summary

# Run and inspect intermediate steps
article = "Tulane is loved for its energy, though some students complain about tuition costs."
summary = logged_sentiment_pipeline(article, verbose=True)
print("🧠 Final Logged Summary:", summary)

🟢 Extracted Sentences: ['Tulane is loved for its energy.', 'Some students complain about tuition costs.']
🟣 Sentiment for 'Tulane is loved for its energy....': positive
🟣 Sentiment for 'Some students complain about tuition costs....': Neutral
🧩 Final Summary: The sentiments expressed include a positive outlook and a neutral perspective, indicating a mixture of optimism and impartiality in the situation.
🧠 Final Logged Summary: The sentiments expressed include a positive outlook and a neutral perspective, indicating a mixture of optimism and impartiality in the situation.



## 📊 Results & Conclusion

Describe what happened when:
- The pipeline ran correctly.
- A step failed or had invalid input.
- You compared the pipeline to the single prompt version.

## Conclusion

- Was your hypothesis supported?  
- How did early-step vs. late-step errors differ?  
- What debugging tools were most helpful?



## 💡 Post-Lab Reflection

1. Which step of the sentiment pipeline was hardest to debug, and why?  
2. How did `safe_run()` change your understanding of reliability in AI systems?  
3. What’s one improvement you could make to this pipeline?


In [10]:
# @title ✅ Run Checks for Lab 3

print("Running Lab 3 checks...")

try:
    from course_utils import safe_run_demo, build_sentiment_pipeline_demo
    result = safe_run_demo("Tulane students are happy.")
    assert result is not None, "safe_run_demo() should return something non-None for valid input"
    print("  ✅ safe_run_demo() works as expected.")

    pipeline = build_sentiment_pipeline_demo()
    summary = pipeline("Tulane is a great place to study.")
    assert isinstance(summary, str)
    print("  ✅ Pipeline returns a string summary.")
except Exception as e:
    print("  ❌ Check failed:", e)

Running Lab 3 checks...
  ✅ safe_run_demo() works as expected.
  ✅ Pipeline returns a string summary.
